# Image Input + Structured Output

Combine multimodal image content blocks (see `l18_image_text.ipynb`) with `llm.with_structured_output(...)` (see `l18_structured_output.ipynb`) to extract a typed Pydantic object directly from an image.

In [1]:
import base64
from pathlib import Path

from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-5.6-luna")

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
def encode_image(path: str) -> str:
    return base64.b64encode(Path(path).read_bytes()).decode("utf-8")

In [6]:
class StudentDetail(BaseModel):
    name:str
    school_name:str
    father_name:str
    dob:str
    address:str
    phone_number:str
    gender:str


In [7]:
llm_schema= llm.with_structured_output(StudentDetail)

In [8]:


message = HumanMessage(
    content=[
        {"type": "text", "text": "extract student details"},
        {
            "type": "image",
            "base64": encode_image("id2.png"),
            "mime_type": "image/png",
        },
    ]
)

result = llm_schema.invoke([message])
result

StudentDetail(name='Hardik Kamboj', school_name='DAV Public School', father_name='Pankaj Kumar', dob='20.06.2014', address='674, Sec-13, Karnal', phone_number='9416655051', gender='Male')

In [9]:
result.name

'Hardik Kamboj'

In [10]:
class PersonDetail(BaseModel):
    name:str
    father_name:str
    id_number:str
    dob:str
    address:str
    gender:str


In [11]:
llm_schema= llm.with_structured_output(PersonDetail)

In [12]:


message = HumanMessage(
    content=[
        {"type": "text", "text": "extract person details"},
        {
            "type": "image",
            "base64": encode_image("ad1.png"),
            "mime_type": "image/png",
        },
    ]
)

result = llm_schema.invoke([message])
result

PersonDetail(name='Vilas Rakhne', father_name='', id_number='7730 0889 2163', dob='30/05/1995', address='ROW H. No. 2, Sonal Nagar, Opp. Chavan Saheb Home, Subhadra Nagar Railway Station, Jalna, PO: Jalna, DIST: Jalna, Maharashtra - 431203', gender='MALE')

In [13]:

result.gender

'MALE'

## Matching the schema to the image

`p3.png` is a tiny 41x41 image — likely an icon or logo. A schema tailored to that kind of content extracts more useful structure than a generic one.

In [3]:
class IconClassification(BaseModel):
    is_icon: bool = Field(description="Whether the image looks like an icon or logo rather than a photo")
    subject: str = Field(description="What the icon/logo depicts")
    style: str = Field(description="Visual style, e.g. flat, 3D, line art, photographic")

In [5]:
icon_llm = llm.with_structured_output(IconClassification)

message = HumanMessage(
    content=[
        {"type": "text", "text": "Classify this small image."},
        {
            "type": "image",
            "base64": encode_image("pc2.png"),
            "mime_type": "image/png",
        },
    ]
)

icon_llm.invoke([message])

IconClassification(is_icon=False, subject='Indian Income Tax Department Permanent Account Number (PAN) identity card', style='photographic document scan')

## Multiple images in one message

A content list can hold more than one image block — useful for comparisons.

In [15]:
class ComparisonResult(BaseModel):
    similarities: list[str] = Field(description="Things the two images have in common")
    differences: list[str] = Field(description="Things that differ between the two images")

comparison_llm = llm.with_structured_output(ComparisonResult)

message = HumanMessage(
    content=[
        {"type": "text", "text": "Compare these two images."},
        {
            "type": "image",
            "base64": encode_image("id1.png"),
            "mime_type": "image/png",
        },
        {
            "type": "image",
            "base64": encode_image("id2.png"),
            "mime_type": "image/png",
        },
    ]
)

comparison_llm.invoke([message])

ComparisonResult(similarities=['Both are school identity cards containing a school logo and school name.', 'Both include the student’s name, class, date of birth, address, and a photograph.', 'Both display school-related personal identification details in a structured text layout.', 'Both include contact or administrative information associated with the school.', 'Both use a colored header and bordered card-style design.'], differences=['The first card is for Jalpai Public School in Jalpaiguri, West Bengal, while the second is for DAV Public School, Sector-13, Karnal, Haryana.', 'The first card identifies the student as Amit Sony; the second identifies the student as Hardik Kamboj.', 'The first student is in Class IX ‘A’; the second is in Nursery.', 'The first card lists the father as Dharmendra Sony and the second lists the father as Pankaj Kumar; the second also includes the mother’s name, Vinay.', 'The first card shows session 2016–17, while the second shows session 2017–18.', 'The 

## When do you need base64?

Same rule as with plain text output: if the image only exists on your side (local file, in-memory bytes, a locally-saved receipt photo), you must inline it as `base64`. If it's already hosted at a reachable URL (e.g. a product image on your CDN), pass `url` instead and skip the encoding step.